# ASHRAE GEPIII — Test set preprocessing

This notebook mirrors `dataset_preprocessing.ipynb`, but processes the **test** split (`test.csv` + `weather_test.csv`) and avoids **train-only** cleaning steps that rely on the target (`meter_reading`) or would remove rows (which would break Kaggle submission alignment by `row_id`).

**Outputs**
- A partitioned parquet dataset directory:
  - `processed_clean/ashrae_test_merged_dataset/` (partitioned by `site_id` and `meter`)


In [2]:
# =========================
# Cell 1 — Setup
# =========================
import os
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.options.mode.chained_assignment = None  # avoid noisy warnings


def df_mem_gb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / (1024**3)


def basic_report(df: pd.DataFrame, name: str, head: int = 3):
    print(f"[{name}] shape={df.shape}  mem={df_mem_gb(df):.3f} GB")
    display(df.head(head))


def missing_report(df: pd.DataFrame, cols=None, topn=30):
    if cols is None:
        cols = df.columns
    s = df[cols].isna().mean().sort_values(ascending=False)
    out = pd.DataFrame({"missing_frac": s, "missing_cnt": (s * len(df)).round().astype("int64")})
    display(out.head(topn))


# ---- Set your Kaggle dataset folder here ----
# It should contain: test.csv, building_metadata.csv, weather_test.csv
DATA_DIR = Path(r"E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction")
assert DATA_DIR.exists(), f"DATA_DIR not found: {DATA_DIR}"

TEST_PATH = DATA_DIR / "test.csv"
BMETA_PATH = DATA_DIR / "building_metadata.csv"
WEATHER_PATH = DATA_DIR / "weather_test.csv"

for p in [TEST_PATH, BMETA_PATH, WEATHER_PATH]:
    assert p.exists(), f"Missing file: {p}"

OUT_DIR = DATA_DIR / "processed_clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_DIR:", DATA_DIR.resolve())
print("OUT_DIR :", OUT_DIR.resolve())

# To keep `ts_idx` consistent between train and test, use the *same* origin as train's minimum timestamp.
# The official train period starts at 2016-01-01 00:00:00.
TS_ORIGIN = pd.Timestamp("2016-01-01 00:00:00")
print("TS_ORIGIN:", TS_ORIGIN)

DATA_DIR: E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction
OUT_DIR : E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_clean
TS_ORIGIN: 2016-01-01 00:00:00


In [3]:
# =========================
# Cell 2 — Load building metadata (light cleaning only)
# =========================
bmeta_dtypes = {
    "site_id": "uint8",
    "building_id": "uint16",
    "primary_use": "category",
    "square_feet": "int32",
    "year_built": "float32",
    "floor_count": "float32",
}

bmeta = pd.read_csv(BMETA_PATH, dtype=bmeta_dtypes)

# conservative validity range: invalid years -> NaN (keep consistent with train preprocessing)
bmeta.loc[
    (bmeta["year_built"].notna()) & ((bmeta["year_built"] < 1900) | (bmeta["year_built"] > 2018)),
    "year_built",
] = np.nan

basic_report(bmeta, "building_metadata")
print("Missingness in building metadata:")
missing_report(bmeta)

[building_metadata] shape=(1449, 6)  mem=0.000 GB


,site_id,building_id,primary_use,square_feet,year_built,floor_count
0,0,0,Education,7432,2008.0,NaN
1,0,1,Education,2720,2004.0,NaN
2,0,2,Education,5376,1991.0,NaN


Missingness in building metadata:


,missing_frac,missing_cnt
floor_count,0.755003,1094
year_built,0.534161,774
building_id,0.000000,0
site_id,0.000000,0
square_feet,0.000000,0
primary_use,0.000000,0


In [4]:
# =========================
# Cell 3 — Load weather_test (pre-merge cleaning)
# =========================
weather_dtypes = {
    "site_id": "uint8",
    "air_temperature": "float32",
    "cloud_coverage": "float32",
    "dew_temperature": "float32",
    "precip_depth_1_hr": "float32",
    "sea_level_pressure": "float32",
    "wind_direction": "float32",
    "wind_speed": "float32",
}

weather = pd.read_csv(WEATHER_PATH, dtype=weather_dtypes, parse_dates=["timestamp"])
basic_report(weather, "weather_test_raw")

print("Weather missingness BEFORE cleaning:")
missing_report(weather)
print("Raw weather row count:", len(weather))
print(
    "Sites:",
    weather["site_id"].nunique(),
    "  Time range:",
    weather["timestamp"].min(),
    "->",
    weather["timestamp"].max(),
)

[weather_test_raw] shape=(277243, 9)  mem=0.010 GB


,site_id,timestamp,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2017-01-01 00:00:00,17.799999,4.0,11.7,NaN,1021.400024,100.0,3.6
1,0,2017-01-01 01:00:00,17.799999,2.0,12.8,0.0,1022.000000,130.0,3.1
2,0,2017-01-01 02:00:00,16.100000,0.0,12.8,0.0,1021.900024,140.0,3.1


Weather missingness BEFORE cleaning:


,missing_frac,missing_cnt
cloud_coverage,0.506588,140448
precip_depth_1_hr,0.344781,95588
sea_level_pressure,0.076702,21265
wind_direction,0.044618,12370
wind_speed,0.001659,460
dew_temperature,0.001179,327
air_temperature,0.000375,104
site_id,0.000000,0
timestamp,0.000000,0


Raw weather row count: 277243
Sites: 16   Time range: 2017-01-01 00:00:00 -> 2018-12-31 23:00:00


In [5]:
# =========================
# Cell 4 — Weather: sentinel cleanup, complete site-hour grid, deterministic imputation, then timestamp alignment
# =========================

# 1) Sentinel cleanup: precip -1 used as missing in many public pipelines
weather.loc[weather["precip_depth_1_hr"] < 0, "precip_depth_1_hr"] = np.nan

# 2) Build full hourly timeline for the weather table (in original timestamp space)
tmin = weather["timestamp"].min()
tmax = weather["timestamp"].max()
full_hours = pd.date_range(tmin, tmax, freq="h")
expected_rows = len(full_hours) * weather["site_id"].nunique()
print("Expected full weather rows (sites * hours):", expected_rows)

cont_cols = ["air_temperature", "dew_temperature", "sea_level_pressure", "wind_speed"]
disc_cols = ["cloud_coverage", "wind_direction"]  # discrete-ish
precip_col = "precip_depth_1_hr"

pieces = []
inserted_rows_total = 0

for site_id, g in weather.groupby("site_id", sort=True):
    g = g.sort_values("timestamp").set_index("timestamp")
    before = len(g)

    g = g.reindex(full_hours)
    inserted = len(g) - before
    inserted_rows_total += inserted

    g["site_id"] = site_id

    # deterministic imputation:
    # - continuous columns: time interpolation + ffill/bfill
    g[cont_cols] = g[cont_cols].interpolate(method="time", limit_direction="both")
    g[cont_cols] = g[cont_cols].ffill().bfill()

    # - discrete-ish columns: ffill/bfill
    g[disc_cols] = g[disc_cols].ffill().bfill()

    # - precip: fill missing with 0 (conservative for "unknown precipitation depth")
    g[precip_col] = g[precip_col].fillna(0.0)

    g = g.reset_index().rename(columns={"index": "timestamp_gmt"})
    pieces.append(g)

weather_full = pd.concat(pieces, ignore_index=True)
print("Inserted missing site-hour rows total:", inserted_rows_total)
basic_report(weather_full, "weather_test_full_gmt")

print("Weather missingness AFTER imputation:")
missing_report(weather_full)

# 3) Timestamp alignment (site_id → hour shift) before merge
# Source mapping: Team Energetic Engineering preprocessing note
timediff = {0: 4, 1: 0, 2: 7, 3: 4, 4: 7, 5: 0, 6: 4, 7: 4, 8: 4, 9: 5, 10: 7, 11: 4, 12: 0, 13: 5, 14: 4, 15: 4}
weather_full["time_diff_hours"] = weather_full["site_id"].map(timediff).astype("int16")

# Align weather to local timestamps used in train/test meter tables
weather_full["timestamp"] = weather_full["timestamp_gmt"] - pd.to_timedelta(weather_full["time_diff_hours"], unit="h")

# Keep only merge-relevant weather columns (+ audit columns)
weather_full = weather_full[
    [
        "site_id",
        "timestamp",
        "timestamp_gmt",
        "time_diff_hours",
        "air_temperature",
        "cloud_coverage",
        "dew_temperature",
        "precip_depth_1_hr",
        "sea_level_pressure",
        "wind_direction",
        "wind_speed",
    ]
].copy()

basic_report(weather_full, "weather_test_aligned_for_merge")
print("Aligned weather time range:", weather_full["timestamp"].min(), "->", weather_full["timestamp"].max())

Expected full weather rows (sites * hours): 280320
Inserted missing site-hour rows total: 3077
[weather_test_full_gmt] shape=(280320, 9)  mem=0.011 GB


,timestamp_gmt,site_id,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,2017-01-01 00:00:00,0,17.799999,4.0,11.7,0.0,1021.400024,100.0,3.6
1,2017-01-01 01:00:00,0,17.799999,2.0,12.8,0.0,1022.000000,130.0,3.1
2,2017-01-01 02:00:00,0,16.100000,0.0,12.8,0.0,1021.900024,140.0,3.1


Weather missingness AFTER imputation:


,missing_frac,missing_cnt
cloud_coverage,0.1250,35040
sea_level_pressure,0.0625,17520
timestamp_gmt,0.0000,0
air_temperature,0.0000,0
site_id,0.0000,0
dew_temperature,0.0000,0
precip_depth_1_hr,0.0000,0
wind_direction,0.0000,0
wind_speed,0.0000,0


[weather_test_aligned_for_merge] shape=(280320, 11)  mem=0.014 GB


,site_id,timestamp,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,2016-12-31 20:00:00,2017-01-01 00:00:00,4,17.799999,4.0,11.7,0.0,1021.400024,100.0,3.6
1,0,2016-12-31 21:00:00,2017-01-01 01:00:00,4,17.799999,2.0,12.8,0.0,1022.000000,130.0,3.1
2,0,2016-12-31 22:00:00,2017-01-01 02:00:00,4,16.100000,0.0,12.8,0.0,1021.900024,140.0,3.1


Aligned weather time range: 2016-12-31 17:00:00 -> 2018-12-31 23:00:00


In [6]:
# =========================
# Cell 5 — Load test (meter data)
# =========================
test_dtypes = {
    "row_id": "int32",
    "building_id": "uint16",
    "meter": "uint8",
}

test = pd.read_csv(TEST_PATH, dtype=test_dtypes, parse_dates=["timestamp"])
basic_report(test, "test_raw")

print("Test time range:", test["timestamp"].min(), "->", test["timestamp"].max())
print("Unique buildings:", test["building_id"].nunique(), "  Unique meters:", test["meter"].nunique())

# NOTE:
# - test.csv has NO meter_reading. Any target-based cleaning must be skipped.
# - Do not drop rows, because Kaggle submission must include every row_id.

# Create compact IDs (kept consistent with train preprocessing)
test["ts_idx"] = ((test["timestamp"] - TS_ORIGIN) / np.timedelta64(1, "h")).astype("int32")
test["pair_id"] = (test["building_id"].astype("int32") * 4 + test["meter"].astype("int32")).astype("int32")

print("ts_idx range:", int(test["ts_idx"].min()), "->", int(test["ts_idx"].max()))

[test_raw] shape=(41697600, 4)  mem=0.583 GB


,row_id,building_id,meter,timestamp
0,0,0,0,2017-01-01
1,1,1,0,2017-01-01
2,2,2,0,2017-01-01


Test time range: 2017-01-01 00:00:00 -> 2018-12-31 23:00:00
Unique buildings: 1449   Unique meters: 4
ts_idx range: 8784 -> 26303


In [7]:
# =========================
# Cell 6 — Optional diagnostics (NO DROPS): timestamp coverage & max gaps per (building_id, meter)
# =========================
# These diagnostics mirror the train notebook's logic, but we only *report* here.
# We do NOT remove any test rows.

tmin = test["ts_idx"].min()
tmax = test["ts_idx"].max()
n_hours_total = int(tmax - tmin + 1)

counts = test.groupby("pair_id", sort=False).size().astype("int32")
missing_ratio = 1.0 - (counts / n_hours_total)

diag = pd.DataFrame(
    {
        "n_rows": counts,
        "missing_ratio_vs_full_range": missing_ratio,
    }
).reset_index()

# Max missing gap (in hours) computed from max ts jump - 1 within each pair
test_sorted = test.sort_values(["pair_id", "ts_idx"], kind="mergesort").reset_index(drop=True)
pair = test_sorted["pair_id"]
ts = test_sorted["ts_idx"]

same_pair = pair.eq(pair.shift(1))
diff = ts.diff()
gap_missing = (diff.where(same_pair, 1) - 1).fillna(0).astype("int32")
max_gap = gap_missing.groupby(pair, sort=False).max().rename("max_missing_gap_hours").reset_index()

diag = diag.merge(max_gap, on="pair_id", how="left")

print("Worst 20 pairs by missing ratio (diagnostic only):")
display(diag.sort_values("missing_ratio_vs_full_range", ascending=False).head(20))

print("Worst 20 pairs by max missing gap (diagnostic only):")
display(diag.sort_values("max_missing_gap_hours", ascending=False).head(20))

Worst 20 pairs by missing ratio (diagnostic only):


,pair_id,n_rows,missing_ratio_vs_full_range,max_missing_gap_hours
0,0,17520,0.0,0
1,4,17520,0.0,0
2,8,17520,0.0,0
3,12,17520,0.0,0
4,16,17520,0.0,0
5,20,17520,0.0,0
6,24,17520,0.0,0
7,28,17520,0.0,0
8,29,17520,0.0,0
9,32,17520,0.0,0


Worst 20 pairs by max missing gap (diagnostic only):


,pair_id,n_rows,missing_ratio_vs_full_range,max_missing_gap_hours
0,0,17520,0.0,0
1,4,17520,0.0,0
2,8,17520,0.0,0
3,12,17520,0.0,0
4,16,17520,0.0,0
5,20,17520,0.0,0
6,24,17520,0.0,0
7,28,17520,0.0,0
8,29,17520,0.0,0
9,32,17520,0.0,0


In [8]:
# =========================
# Cell 7 — Merge: test + building_metadata + weather (aligned timestamps)
# =========================
# Merge building metadata first (gives site_id for weather join)
df = test.merge(bmeta, on="building_id", how="left", validate="many_to_one")

# Merge weather on (site_id, timestamp)
df = df.merge(weather_full, on=["site_id", "timestamp"], how="left", validate="many_to_one")

basic_report(df, "merged_test_df", head=5)

print("Missingness after merge (key fields):")
missing_report(
    df,
    cols=[
        "site_id",
        "primary_use",
        "square_feet",
        "air_temperature",
        "dew_temperature",
        "precip_depth_1_hr",
        "sea_level_pressure",
        "wind_speed",
        "cloud_coverage",
        "wind_direction",
    ],
)

[merged_test_df] shape=(41697600, 20)  mem=3.146 GB


,row_id,building_id,meter,timestamp,ts_idx,pair_id,site_id,primary_use,square_feet,year_built,floor_count,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,0,2017-01-01,8784,0,0,Education,7432,2008.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
1,1,1,0,2017-01-01,8784,4,0,Education,2720,2004.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
2,2,2,0,2017-01-01,8784,8,0,Education,5376,1991.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
3,3,3,0,2017-01-01,8784,12,0,Education,23685,2002.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
4,4,4,0,2017-01-01,8784,16,0,Education,116607,1975.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6


Missingness after merge (key fields):


,missing_frac,missing_cnt
sea_level_pressure,0.037651,1569953
cloud_coverage,0.023780,991569
wind_speed,0.000256,10673
precip_depth_1_hr,0.000256,10673
air_temperature,0.000256,10673
dew_temperature,0.000256,10673
wind_direction,0.000256,10673
site_id,0.000000,0
primary_use,0.000000,0
square_feet,0.000000,0


In [9]:
# =========================
# Cell 8 — Handle rare post-merge missingness (NO DROPS)
# =========================
# Train preprocessing *dropped* rows with missing critical fields (meter_reading/square_feet/air_temperature).
# For test, we keep all rows and apply deterministic imputations only when needed.

# 1) Building metadata sanity
if df["site_id"].isna().any():
    # If this happens, building_id not found in building_metadata.csv.
    # This should not occur for the official GEPIII dataset; treat as a hard error to keep schema consistent.
    missing_blds = df.loc[df["site_id"].isna(), "building_id"].unique()[:20]
    raise ValueError(f"Missing site_id after building_metadata merge. Example building_ids: {missing_blds}")

# square_feet should almost always be present; if not, fill with global median from bmeta (keeps all rows)
if df["square_feet"].isna().any():
    sqft_med = int(bmeta["square_feet"].median())
    print(f"WARNING: missing square_feet. Filling with bmeta median (int): {sqft_med}")
    df["square_feet"] = df["square_feet"].fillna(sqft_med)

# Cast back to int32 for consistency with train
df["square_feet"] = df["square_feet"].astype("int32", copy=False)

# 2) Weather post-merge sanity & fills
weather_cols = [
    "air_temperature",
    "cloud_coverage",
    "dew_temperature",
    "precip_depth_1_hr",
    "sea_level_pressure",
    "wind_direction",
    "wind_speed",
]

missing_any_weather = df[weather_cols].isna().any().any()
if missing_any_weather:
    print("WARNING: missing weather values after merge. Applying deterministic fills.")
    # site-level medians computed from the aligned test weather table (independent of target)
    site_medians = weather_full.groupby("site_id")[weather_cols].median(numeric_only=True)
    global_medians = weather_full[weather_cols].median(numeric_only=True)

    for c in weather_cols:
        df[c] = df[c].fillna(df["site_id"].map(site_medians[c]))
        df[c] = df[c].fillna(global_medians[c])

# Ensure float32 for weather columns (keeps memory down and matches train)
for c in weather_cols:
    df[c] = df[c].astype("float32", copy=False)

print("Missingness snapshot AFTER deterministic fills:")
missing_report(df, cols=["square_feet"] + weather_cols)

C:\Users\zpz\AppData\Local\Temp\ipykernel_23844\806992557.py:21: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df["square_feet"] = df["square_feet"].astype("int32", copy=False)


C:\Users\zpz\AppData\Local\Temp\ipykernel_23844\806992557.py:47: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df[c] = df[c].astype("float32", copy=False)
C:\Users\zpz\AppData\Local\Temp\ipykernel_23844\806992557.py:47: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Use .copy() to make an eager copy if necessary.
  df[c] = df[c].astype("float32", copy=False)
C:\Users\zpz\AppData\Local\Temp\ipykernel_23844\806992557.py:47: Pandas4Warning: The copy keyword is deprecated and will be removed in a future version. Copy-on-Write is active in pandas since 3.0 which utilizes a lazy copy mechanism that defers copies until necessary. Us

Missingness snapshot AFTER deterministic fills:


,missing_frac,missing_cnt
square_feet,0.0,0
air_temperature,0.0,0
cloud_coverage,0.0,0
dew_temperature,0.0,0
precip_depth_1_hr,0.0,0
sea_level_pressure,0.0,0
wind_direction,0.0,0
wind_speed,0.0,0


In [10]:
# =========================
# Cell 9 — Final sanity reports
# =========================
basic_report(df, "FINAL_TEST_DF", head=5)

print("Final missingness snapshot (top):")
missing_report(df)

print("Row count (must equal test.csv rows):", len(df))
print("Unique row_id:", df["row_id"].nunique())
assert len(df) == df["row_id"].nunique(), "row_id must remain unique (no drops/duplication)."

[FINAL_TEST_DF] shape=(41697600, 20)  mem=3.146 GB


,row_id,building_id,meter,timestamp,ts_idx,pair_id,site_id,primary_use,square_feet,year_built,floor_count,timestamp_gmt,time_diff_hours,air_temperature,cloud_coverage,dew_temperature,precip_depth_1_hr,sea_level_pressure,wind_direction,wind_speed
0,0,0,0,2017-01-01,8784,0,0,Education,7432,2008.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
1,1,1,0,2017-01-01,8784,4,0,Education,2720,2004.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
2,2,2,0,2017-01-01,8784,8,0,Education,5376,1991.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
3,3,3,0,2017-01-01,8784,12,0,Education,23685,2002.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6
4,4,4,0,2017-01-01,8784,16,0,Education,116607,1975.0,NaN,2017-01-01 04:00:00,4.0,16.700001,2.0,13.3,0.0,1022.299988,130.0,2.6


Final missingness snapshot (top):


,missing_frac,missing_cnt
floor_count,0.826050,34444320
year_built,0.589916,24598080
time_diff_hours,0.000256,10673
timestamp_gmt,0.000256,10673
timestamp,0.000000,0
meter,0.000000,0
building_id,0.000000,0
row_id,0.000000,0
primary_use,0.000000,0
site_id,0.000000,0


Row count (must equal test.csv rows): 41697600
Unique row_id: 41697600


In [11]:
# =========================
# Cell 10 — Save as Parquet (partitioned dataset)
# =========================
out_dataset_dir = OUT_DIR / "ashrae_test_merged_dataset"
print("Writing parquet dataset to:", out_dataset_dir)

if out_dataset_dir.exists():
    raise FileExistsError(f"Output directory already exists: {out_dataset_dir}")

out_dataset_dir.mkdir(exist_ok=False)

df.to_parquet(
    out_dataset_dir,
    index=False,
    engine="pyarrow",
    compression="snappy",
    partition_cols=["site_id", "meter"],
)

print("Done. Parquet dataset directory:", out_dataset_dir.resolve())

Writing parquet dataset to: E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_clean\ashrae_test_merged_dataset
Done. Parquet dataset directory: E:\repos\LLM_traffic_query\tests\energy_prediction\dataset\ashrae-energy-prediction\processed_clean\ashrae_test_merged_dataset


## Train-only steps that are intentionally skipped for test

The following steps from `dataset_preprocessing.ipynb` are **not applied** to `test.csv` because they either require the target (`meter_reading`) or would remove rows and break `row_id` alignment for Kaggle submission:

1. **Dropping negative `meter_reading` and duplicate (building_id, meter, timestamp) keys**
   - `test.csv` has no `meter_reading`. Also, dropping duplicate keys would still remove rows and you must submit a prediction for every `row_id`.

2. **Dropping entire (building_id, meter) pairs based on missing timestamp coverage (>50%) or large max gaps (>=2400h)**
   - Those rules are target-quality heuristics meant for *training*. For test, you must predict for every row.

3. **Corrections and drops that directly reference `meter_reading`**
   - Site 0 electricity unit correction multiplies `meter_reading` (target) and is not applicable in test preprocessing.
   - The “early-year zero block” removal and the “>=30 consecutive days of zeros” rule both rely on observed `meter_reading` patterns.

4. **Dropping specific outlier buildings (e.g., 1099, 778)**
   - For test, you cannot remove building IDs; you must produce predictions for them.

In this test notebook we keep rows intact and, if needed, only apply **deterministic imputations** for merge-time missing values.
